# Sales Forecasting Using LSTM with PyTorch

**Goal:** use the Kaggle Store Sales - Time Series Forecasting data to learn historical daily sales patterns, evaluate an LSTM, and forecast the **next 15 days**.

The project brief asks to explore/prepare the data, create LSTM sequences, build and train an LSTM in PyTorch, evaluate it, compare predictions with actual values, visualize the results, and generate future sales predictions. 

Kaggle source: https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data

In [ ]:
# 1. Install and import libraries
!pip -q install torch scikit-learn pandas numpy matplotlib

import os, zipfile, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Upload the Kaggle data

Download the ZIP from the supplied Kaggle page, then run the next cell and upload it to Colab.

The Kaggle competition provides historical `train.csv` and future `test.csv`; the test dates are the future period to predict. 

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
# 3. Extract and locate train.csv / test.csv
zip_files = list(Path("/content").glob("*.zip"))
if zip_files:
    with zipfile.ZipFile(zip_files[0], "r") as z:
        z.extractall("/content/store_sales_data")

train_files = list(Path("/content").rglob("train.csv"))
test_files = list(Path("/content").rglob("test.csv"))

if not train_files or not test_files:
    raise FileNotFoundError("Please upload the Kaggle ZIP containing train.csv and test.csv.")

TRAIN_PATH = train_files[0]
TEST_PATH = test_files[0]
print("Train:", TRAIN_PATH)
print("Test :", TEST_PATH)

In [ ]:
# 4. Load and inspect
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Train dates:", train.date.min().date(), "to", train.date.max().date())
print("Test dates :", test.date.min().date(), "to", test.date.max().date())
display(train.head())
display(test.head())

## 3. Prepare the daily time series

The raw competition data contains sales by store and product family. For this project, we aggregate `sales` across all stores and families to create one overall daily sales series, making the objective clear: **predict total sales for the coming days**.

In [ ]:
# 5. Aggregate sales by date
daily = train.groupby("date", as_index=False)["sales"].sum().sort_values("date")
full_dates = pd.date_range(daily.date.min(), daily.date.max(), freq="D")
daily = daily.set_index("date").reindex(full_dates).rename_axis("date").reset_index()
daily["sales"] = daily["sales"].interpolate().fillna(0)

print("Daily observations:", len(daily))
display(daily.head())
display(daily.tail())

plt.figure(figsize=(14,5))
plt.plot(daily.date, daily.sales)
plt.title("Historical Daily Sales")
plt.xlabel("Date"); plt.ylabel("Total Sales"); plt.grid(alpha=.3)
plt.show()

## 4. Create sequences and a validation period

We hold out the last **15 historical days** so the model can be tested against real known sales before forecasting the truly future dates. The LSTM uses the previous 30 days to predict the next day.

In [ ]:
# 6. Split data and scale only the training part
FORECAST_HORIZON = 15
LOOKBACK = 30

values = daily.sales.values.astype(np.float32)
train_values = values[:-FORECAST_HORIZON]
val_values = values[-FORECAST_HORIZON:]

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_values.reshape(-1,1)).flatten()

def make_sequences(series, lookback):
    X, y = [], []
    for i in range(lookback, len(series)):
        X.append(series[i-lookback:i])
        y.append(series[i])
    return np.array(X, np.float32).reshape(-1,lookback,1), np.array(y, np.float32).reshape(-1,1)

X_train, y_train = make_sequences(train_scaled, LOOKBACK)

class SalesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

loader = DataLoader(SalesDataset(X_train,y_train), batch_size=64, shuffle=True)
print("X_train:", X_train.shape, "y_train:", y_train.shape)

## 5. Build the LSTM model

In [ ]:
class SalesLSTM(nn.Module):
    def __init__(self, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden_size, num_layers,
                            batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = SalesLSTM().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(model)

In [ ]:
# 7. Train
EPOCHS = 30
losses = []

for epoch in range(EPOCHS):
    model.train()
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)

    epoch_loss = total / len(loader.dataset)
    losses.append(epoch_loss)
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {epoch_loss:.6f}")

plt.figure(figsize=(10,4))
plt.plot(losses)
plt.title("Training Loss")
plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.grid(alpha=.3)
plt.show()

## 6. Evaluate on the last 15 known days

In [ ]:
def inverse_scale(x, fitted_scaler):
    return fitted_scaler.inverse_transform(np.asarray(x).reshape(-1,1)).flatten()

history = list(train_scaled[-LOOKBACK:])
pred_scaled = []

model.eval()
with torch.no_grad():
    for _ in range(FORECAST_HORIZON):
        xb = torch.tensor(history[-LOOKBACK:], dtype=torch.float32).view(1,LOOKBACK,1).to(device)
        p = model(xb).item()
        pred_scaled.append(p)
        history.append(p)

val_pred = np.maximum(inverse_scale(pred_scaled, scaler), 0)
mae = mean_absolute_error(val_values, val_pred)
rmse = mean_squared_error(val_values, val_pred) ** 0.5

comparison = pd.DataFrame({
    "date": daily.date.iloc[-FORECAST_HORIZON:].values,
    "actual_sales": val_values,
    "predicted_sales": val_pred
})

print(f"Validation MAE : {mae:,.2f}")
print(f"Validation RMSE: {rmse:,.2f}")
display(comparison)

In [ ]:
# 8. Actual vs predicted
plt.figure(figsize=(14,5))
plt.plot(comparison.date, comparison.actual_sales, marker="o", label="Actual")
plt.plot(comparison.date, comparison.predicted_sales, marker="o", label="Predicted")
plt.title("Validation: Actual vs Predicted Sales")
plt.xlabel("Date"); plt.ylabel("Sales")
plt.legend(); plt.grid(alpha=.3); plt.xticks(rotation=45)
plt.show()

## 7. Retrain on all historical data and forecast the next 15 days

After evaluation, the final model uses all historical observations and generates a forecast for the future dates in `test.csv`.

In [ ]:
# 9. Final training on all historical data
final_scaler = MinMaxScaler()
all_scaled = final_scaler.fit_transform(values.reshape(-1,1)).flatten()
X_all, y_all = make_sequences(all_scaled, LOOKBACK)
all_loader = DataLoader(SalesDataset(X_all,y_all), batch_size=64, shuffle=True)

final_model = SalesLSTM().to(device)
final_optimizer = torch.optim.Adam(final_model.parameters(), lr=0.001)

for epoch in range(30):
    final_model.train()
    for xb, yb in all_loader:
        xb, yb = xb.to(device), yb.to(device)
        final_optimizer.zero_grad()
        loss = criterion(final_model(xb), yb)
        loss.backward()
        final_optimizer.step()
    if (epoch+1) % 5 == 0:
        print("Final epoch:", epoch+1)

In [ ]:
# 10. Forecast future dates
future_dates = pd.to_datetime(test.date.drop_duplicates().sort_values()).reset_index(drop=True)
history = list(all_scaled[-LOOKBACK:])
future_scaled = []

final_model.eval()
with torch.no_grad():
    for _ in range(len(future_dates)):
        xb = torch.tensor(history[-LOOKBACK:], dtype=torch.float32).view(1,LOOKBACK,1).to(device)
        p = final_model(xb).item()
        future_scaled.append(p)
        history.append(p)

future_pred = np.maximum(
    final_scaler.inverse_transform(np.array(future_scaled).reshape(-1,1)).flatten(), 0
)

forecast = pd.DataFrame({
    "date": future_dates,
    "predicted_total_sales": future_pred
})

display(forecast)

In [ ]:
# 11. Plot the future forecast
plt.figure(figsize=(14,5))
plt.plot(daily.date.iloc[-90:], daily.sales.iloc[-90:], label="Historical")
plt.plot(forecast.date, forecast.predicted_total_sales, marker="o", label="Forecast")
plt.title("Next 15 Days Sales Forecast")
plt.xlabel("Date"); plt.ylabel("Total Sales")
plt.legend(); plt.grid(alpha=.3); plt.xticks(rotation=45)
plt.show()

In [ ]:
# 12. Save the results
forecast.to_csv("/content/next_15_days_sales_forecast.csv", index=False)
comparison.to_csv("/content/validation_actual_vs_predicted.csv", index=False)
torch.save(final_model.state_dict(), "/content/sales_lstm_pytorch.pth")

print("Saved successfully:")
print("1) next_15_days_sales_forecast.csv")
print("2) validation_actual_vs_predicted.csv")
print("3) sales_lstm_pytorch.pth")

# Source notebook status

This preserved source has no completed training/evaluation outputs. Run the revised notebook for measured evidence. It forecasts aggregate daily sales, not a Kaggle store × family submission.